# Análisis de textos

---

El *Procesamiento de Lenguaje Natural (NLP)* es una rama de la inteligencia artificial que permite a las computadoras entender, interpretar y generar lenguaje humano. Su objetivo es transformar textos o discursos en datos que una máquina pueda analizar, facilitando tareas como traducción automática, chatbots, análisis de sentimientos o motores de búsqueda. 📖🤖


## Setup

---

In [ ]:
#!pip install wordcloud spacy torch transformers

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

import spacy


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

In [ ]:
model_name = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

In [ ]:
#spacy.cli.download("es_core_news_sm")
nlp = spacy.load("es_core_news_sm")

In [ ]:
sns.set_style('whitegrid')
sns.set_palette('viridis')

##  Vectorización de textos

---

La *vectorización de textos* es el proceso de transformar palabras o frases en representaciones numéricas que las computadoras puedan procesar. 📊 En lugar de trabajar con texto plano, lo convertimos en vectores que permiten a los algoritmos de machine learning analizar, comparar y encontrar patrones en el lenguaje. ✨

Para comprender mejor el concepto  Imagina que una empresa de cosméticos acaba de lanzar al mercado una nueva **crema antimanchas** con vitamina C, pensada para mejorar el tono de la piel y reducir imperfecciones. 🧴✨  

Después de varias semanas, la compañía empieza a recibir comentarios de clientes en diferentes canales: algunos destacan los beneficios (piel más luminosa, aroma agradable, sensación de frescura), mientras que otros muestran inconformidad (irritación, resequedad, poco efecto visible).  

En este contexto, el área de datos ha sido encargada de **analizar las opiniones de los usuarios** para obtener información útil:  
* Identificar si la mayoría de comentarios son positivos o negativos.  
* Reconocer palabras clave que se repiten en las reseñas.  
* Preparar un conjunto de datos limpio para entrenar un modelo de clasificación de sentimientos.  

Este ejercicio nos servirá como un ejemplo práctico de cómo aplicar técnicas de **Procesamiento de Lenguaje Natural (NLP)**, desde la limpieza de texto hasta la vectorización (BoW, TF-IDF, embeddings) y la construcción de un modelo sencillo que clasifique las opiniones como *positivas* o *negativas*. 📊🤖


<center>
  <img src="https://i5.walmartimages.com.mx/gr/images/product-images/img_large/00360054263068L.jpg" alt="Regresion Descenso de Gradiente" style="max-width:50%; height:auto;"  width="30%">
</center>

**Cargado de los datos y análisis exploratorio**

In [ ]:
opiniones=pd.read_csv("https://raw.githubusercontent.com/zyntonyson/bootcamp_ds_da/refs/heads/main/16-ds-analisis-textos/opiniones_garnier.csv")
opiniones

In [ ]:
opiniones.shape, opiniones.Label.value_counts()

In [ ]:
# Longitud de comentarios y distribución
opiniones["len"] = opiniones["Comentario"].str.len()
opiniones["n_words"] = opiniones["Comentario"].apply(lambda x: len(x.split()))
opiniones.groupby("Label")["len"].describe()

In [ ]:
#Cantidad de palabras
opiniones.groupby("Label")["n_words"].describe()

In [ ]:


sns.countplot(data=opiniones, x="Label", order=["positiva","negativa"])
plt.title("Distribución de clases")
plt.show()

sns.boxplot(data=opiniones, x="Label", y="len", order=["positiva","negativa"])
plt.title("Longitud del texto por clase")
plt.show()

sns.boxplot(data=opiniones, x="Label", y="n_words", order=["positiva","negativa"])
plt.title("Longitud del texto por clase")
plt.show()

**Wordclouds**

Las *wordclouds* o nubes de palabras son una herramienta visual que permite identificar de forma rápida las palabras más frecuentes dentro de un conjunto de textos. ☁️✨ En ellas, el tamaño de cada término refleja su relevancia o frecuencia en los datos, lo que ayuda a detectar patrones, temas recurrentes o palabras clave en reseñas, comentarios o documentos sin necesidad de leerlos uno por uno. Son especialmente útiles en etapas iniciales de análisis exploratorio para tener una primera impresión del lenguaje que más utilizan los usuarios. 🔎


In [ ]:
stop_es = set(STOPWORDS) | {"de","la","el","los","las","que","y","mi","me","lo","es"}

def nubes_por_clase(df, label):
    texto = " ".join(df.loc[df.Label==label, "Comentario"].astype(str))
    wc = WordCloud(width=800, height=400, stopwords=stop_es, background_color="white").generate(texto)
    plt.figure(figsize=(8,4)); plt.imshow(wc); plt.axis("off"); plt.title(f"WordCloud: {label}"); plt.show()

for lab in ["positiva","negativa"]:
    nubes_por_clase(opiniones, lab)

**Lematizacion**

La *lematización* es una técnica de procesamiento de texto que consiste en reducir cada palabra a su forma base o “lema”. 🔤 Por ejemplo, las palabras *corriendo*, *corrí* y *correrá* se transforman en su raíz común: *correr*. De esta manera, se unifica el vocabulario y se evita que el modelo considere como diferentes a términos que en realidad expresan la misma idea. Esto mejora la calidad del análisis, pues disminuye la dispersión de palabras y facilita la detección de patrones en los textos. 📊🤖


In [ ]:
nlp = spacy.load("es_core_news_sm")

doc = nlp("Los estudiantes corriendo aprendieron rápidamente")
print([token.lemma_ for token in doc])

In [ ]:
def lematizar(texto):
    doc = nlp(texto.lower())
    # quitamos puntuación, espacios y stopwords del modelo
    lemmas = [t.lemma_ for t in doc if not (t.is_punct or t.is_space or t.is_stop)]
    return " ".join(lemmas)

opiniones["Comentario_lem"] = opiniones["Comentario"].astype(str).apply(lematizar)
opiniones[["Comentario","Comentario_lem"]].head()

**Bolsa de palabras** `BoW`

La *Bolsa de Palabras (Bag of Words, BoW)* es una técnica simple pero poderosa para representar textos en forma numérica. 📊 Consiste en construir un vocabulario con todas las palabras únicas de un corpus y luego transformar cada documento en un vector que indica cuántas veces aparece cada palabra, sin importar el orden en que se presenten.  

Por ejemplo, con las frases *"me gusta la crema"* y *"la crema no funciona"*, el vocabulario sería:  
$ V = \{me, gusta, la, crema, no, funciona\} $  

Cada documento se representa como:  
- *"me gusta la crema"* → $[1, 1, 1, 1, 0, 0]$  
- *"la crema no funciona"* → $[0, 0, 1, 1, 1, 1]$  

De forma general, si $V = \{t_1, t_2, ..., t_n\}$ es el vocabulario y $d_j$ un documento, entonces el vector resultante se define como:  

$$
\text{BoW}(d_j) = \big( f(t_1, d_j), f(t_2, d_j), \dots, f(t_n, d_j) \big)
$$

donde $f(t_i, d_j)$ es la frecuencia de la palabra $t_i$ en el documento $d_j$.  

Aunque no captura el contexto ni el orden de las palabras, la BoW es un excelente punto de partida para modelos de clasificación de texto y análisis de sentimientos. 🤖✨


In [ ]:
cv = CountVectorizer(ngram_range=(1,2), min_df=1)
X_bow = cv.fit_transform(opiniones["Comentario"])
y = opiniones["Label"].apply(lambda x: int(x=='positiva'))

cv.get_feature_names_out()[:20], X_bow.shape

In [ ]:
# 
X_train, X_test, y_train, y_test = train_test_split(X_bow, y, test_size=0.4, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=200, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))
print(confusion_matrix(y_test, y_pred))

**Term Frequency – Inverse Document Frequency**

La técnica de *TF-IDF (Term Frequency – Inverse Document Frequency)* es una mejora sobre la Bolsa de Palabras que busca no solo contar palabras, sino también **ponderar su importancia** dentro de un conjunto de documentos. 📊  

Se basa en dos componentes:  

1. **Frecuencia de término (TF):** mide cuántas veces aparece una palabra en un documento.  
$$
TF(t, d) = \frac{f(t,d)}{\sum_{k} f(k,d)}
$$  
donde $f(t,d)$ es la frecuencia del término $t$ en el documento $d$.  

2. **Frecuencia inversa de documento (IDF):** mide qué tan “única” es una palabra en la colección de textos.  
$$
IDF(t) = \log \frac{N}{1 + DF(t)}
$$  
donde $N$ es el número total de documentos y $DF(t)$ es el número de documentos que contienen el término $t$.  

La puntuación final se calcula como:  
$$
TFIDF(t,d) = TF(t,d) \times IDF(t)
$$  

👉 Con esto, palabras comunes como *"la"*, *"de"* o *"el"* reciben un peso bajo, mientras que términos más específicos como *"manchas"*, *"vitamina"* o *"irritación"* adquieren un mayor valor. Esto ayuda a que los modelos de machine learning se enfoquen en las palabras realmente distintivas para diferenciar sentimientos o temas en los textos. ✨


In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=1)
X_tfidf = tfidf.fit_transform(opiniones["Comentario"])
X_tfidf.shape

In [ ]:
# Cambia X_bow por X_tfidf si quieres usar TF-IDF
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.4, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=200, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))
print(confusion_matrix(y_test, y_pred))

**Embbedings**

Los *embeddings* son representaciones vectoriales densas de palabras que capturan no solo su presencia, sino también su **significado y relaciones semánticas** en un espacio continuo. 🔤➡️📊 A diferencia de BoW o TF-IDF, donde cada palabra es independiente, los embeddings permiten que palabras con significados similares tengan vectores cercanos entre sí.  

Un ejemplo clásico es *Word2Vec*, que aprende estas representaciones a partir del contexto en el que aparecen las palabras, utilizando arquitecturas como **CBOW** (Continuous Bag of Words) o **Skip-gram**. Así, términos como *“bueno”* y *“excelente”* terminan representados con vectores próximos, mientras que *“caro”* y *“costoso”* también se agrupan.  

En Python, a través de `scipy` y librerías afines como `gensim`, se pueden aprovechar embeddings preentrenados y cálculos de similitud entre vectores. Algunas opciones disponibles son:  

* **Word2Vec** (entrenado en grandes corpus de texto).  
* **fastText** (maneja mejor palabras desconocidas gracias a sub-palabras).  
* **GloVe** (Global Vectors for Word Representation, entrenado en co-ocurrencias).  

En `scipy` es común usar funciones como `scipy.spatial.distance.cosine` para medir la similitud entre vectores de palabras generados con estas técnicas.  

👉 Los embeddings marcan un gran avance porque permiten que los modelos entiendan relaciones más profundas en el lenguaje, y son la base de modelos más complejos como **BERT** o **GPT**. 🤖✨


In [ ]:
def get_embbeding(text):
    doc=nlp(text.lower())
    return doc.vector

In [ ]:
X_embbeding=np.stack(opiniones['Comentario'].apply(get_embbeding).values)

In [ ]:
# Cambia X_bow por X_tfidf si quieres usar TF-IDF
X_train, X_test, y_train, y_test = train_test_split(X_embbeding, y, test_size=0.4, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=200, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))
print(confusion_matrix(y_test, y_pred))

**BERT embbedings**

Los *embeddings de BERT* representan una evolución frente a técnicas clásicas como Word2Vec o TF-IDF, ya que son **contextuales**: la representación de una palabra depende de las palabras que la rodean. 📖✨ Por ejemplo, la palabra *“banco”* tendrá un vector diferente en *“me senté en el banco”* que en *“trabajo en un banco”*, porque el modelo entiende el contexto semántico.  

BERT (*Bidirectional Encoder Representations from Transformers*) utiliza la arquitectura *Transformer* y se entrena leyendo textos en ambas direcciones (izquierda y derecha), lo que le permite capturar relaciones complejas entre palabras. Los embeddings que genera son vectores densos de alta dimensión (normalmente 768), que pueden representar tanto palabras individuales como frases o documentos completos.  

Estos embeddings se utilizan como entrada en múltiples tareas de NLP: clasificación de sentimientos, análisis de temas, reconocimiento de entidades o búsqueda semántica. En Python, con la librería `transformers` de HuggingFace, es posible cargar modelos preentrenados en español (como **BETO**) y generar embeddings listos para alimentar a clasificadores tradicionales o para hacer *fine-tuning* en tareas específicas. 🚀🤖


In [ ]:
def get_BERT_embedding(texto):
    inputs = tokenizer(texto, return_tensors="pt", truncation=True, padding=True, max_length=50)
    with torch.no_grad():
        outputs = model(**inputs)
    # Tomamos la representación de la última capa oculta (batch_size, seq_len, hidden_size)
    embeddings = outputs.last_hidden_state
    # Promediamos sobre la secuencia → vector fijo (hidden_size=768)
    vector = embeddings.mean(dim=1).squeeze().numpy()
    return vector

In [ ]:
X_BERT=np.vstack(opiniones["Comentario"].apply(get_BERT_embedding).values)

In [ ]:
# Cambia X_bow por X_tfidf si quieres usar TF-IDF
X_train, X_test, y_train, y_test = train_test_split(X_BERT, y, test_size=0.4, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=200, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))
print(confusion_matrix(y_test, y_pred))

## Para cerrar 💬🤔

----

1. Explica en tus palabras los conceptos de:
    - Bolsa de palabras
    - Tf-IdF
    - Embbedings


## 🚀 Para seguir aprendiendo :

---

- 📚 Vuelve a revisar este notebook y trata resolver por tu cuenta el proyecto nuevamente
- 💬 Recuerda que en Discord puedes dejar todos tus comentarios y dudas sobre el contenido del sprint en [Sprint 15](https://discord.com/channels/1081207584104656986/1270074296395497513).
    - 📝 Si tienes preguntas sobre tu proyecto, usa el canal `#project` para recibir ayuda y compartir ideas.
    - 🤝 Aprovecha el espacio de `CoLearning` para aclarar tus dudas junto con otros estudiantes e instructores: [Co-Learning](https://discord.com/channels/1081207584104656986/1197953851391746119).
- 📅 ¿Necesitas ayuda personalizada? Puedes agendar una sesión `1:1` conmigo aquí: [1:1 Roman Castillo](https://scheduler.zoom.us/roman-castillo/1-1-roman-castillo).

- Por último hazme paro y responde la encuesta al final de la sesión, me sirve para poder ayudarte mejor 

¡Sigue practicando y no dudes en pedir apoyo cuando lo necesites! 💪✨